# Part 2 — Probability: the language of clouds

_Rigorous Courses · Diffusion Models — Part 2 of 12_

**Random variables, joint tables, Bayes' rule, expectation, and variance — the toolkit every later part speaks in**

Every rule the lesson derived on paper gets checked numerically here. You will watch an empirical PMF converge to the true table, confirm that a density of 2 is perfectly legal, slice a joint table into marginals and conditionals, run the medical-test Bayes flip on simulated patients, and see with your own eyes when variances add — and when they refuse to.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## Random variables: numbers with a spread

A **random variable** is a number whose value is decided by chance. Its behavior is fully described by its **PMF** — a table $p(x)$ obeying two rules: $p(x) \ge 0$ for every value, and $\sum_x p(x) = 1$.

The lesson claims that probabilities are *long-run frequencies*: roll a die forever and each face shows up $1/6$ of the time. A computer lets us test that claim directly — roll a simulated die and compare the observed fractions to the true table.

### Step 1 — Roll a simulated die 60 times

`rng.integers(1, 7, size=60)` draws 60 values uniformly from {1, 2, 3, 4, 5, 6} (the upper end 7 is excluded). With only 60 rolls, expect the empirical fractions to be *near* 1/6 ≈ 0.167 but visibly rough — chance has not had room to average out yet.

In [ ]:
n_rolls = 60
rolls = rng.integers(1, 7, size=n_rolls)
values = np.arange(1, 7)
counts = np.array([(rolls == v).sum() for v in values])
empirical_pmf = counts / n_rolls

print("value        :", values)
print("count        :", counts)
print("empirical PMF:", np.round(empirical_pmf, 3))
print("true PMF     :", np.round(np.ones(6) / 6, 3))

### Step 2 — Watch the empirical PMF converge to the true PMF

Now repeat with 100, then 10,000, then 1,000,000 rolls. The bars should hug the dashed 1/6 line more and more tightly. This is the "probability = long-run fraction" idea becoming visible: more repetitions, less roughness.

In [ ]:
sample_sizes = [100, 10_000, 1_000_000]
max_devs = []

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, n in zip(axes, sample_sizes):
    many_rolls = rng.integers(1, 7, size=n)
    pmf_hat = np.array([(many_rolls == v).mean() for v in values])
    max_devs.append(np.abs(pmf_hat - 1 / 6).max())
    ax.bar(values, pmf_hat, color="#4ea1ff")
    ax.axhline(1 / 6, color="#ff7b72", linestyle="--", label="true 1/6")
    ax.set_title(f"n = {n:,}")
    ax.set_xlabel("die value")
    ax.set_ylabel("empirical probability")
    ax.legend()
plt.tight_layout()
plt.show()

print("max |empirical - true| at each n:", np.round(max_devs, 4))

assert max_devs[0] > max_devs[2], "more rolls should track the true PMF more closely"
assert max_devs[2] < 0.005, "with a million rolls, every face frequency should be within 0.005 of 1/6"

## From tables to curves: continuous variables

A spinner landing anywhere in an interval has *infinitely many* outcomes, so no table can describe it. Instead we use a **density** $p(x)$: probability per unit length. Probability lives on intervals, as area under the curve:

$$P(a \le X \le b) = \int_a^b p(x)\, dx$$

The lesson's example: uniform on $[0, 1/2]$. The base is only $1/2$ wide, and the total area must be 1, so the height is **2**. A density above 1 is legal — it is a rate, not a probability. Both claims are testable.

### Step 3 — Sample the spinner and see density 2

We draw 100,000 samples from Uniform(0, 1/2) and plot a *density-normalized* histogram (`density=True` divides each bar by the total count and the bar width, turning counts into estimated density). The bars should sit at height 2. We also check an interval probability against the area rule: P(0.1 ≤ X ≤ 0.3) should be 2 × 0.2 = 0.4. And note how many samples hit *exactly* 0.25 — the lesson says individual points carry zero probability.

In [ ]:
n_samples = 100_000
samples = rng.uniform(0, 0.5, size=n_samples)

plt.figure(figsize=(7, 4))
plt.hist(samples, bins=50, density=True, color="#4ea1ff", label="histogram of samples")
plt.axhline(2.0, color="#ff7b72", linestyle="--", label="density p(x) = 2")
plt.xlabel("x")
plt.ylabel("density (probability per unit length)")
plt.title("Uniform on [0, 1/2]: the density is 2, and that is fine")
plt.legend()
plt.show()

in_window = ((samples >= 0.1) & (samples <= 0.3)).mean()
exact_hits = (samples == 0.25).sum()

print(f"fraction of samples in [0.1, 0.3]: {in_window:.4f}   (area prediction: 2 x 0.2 = 0.4)")
print(f"samples equal to exactly 0.25    : {exact_hits}")

assert abs(in_window - 0.4) < 0.01, "an interval's probability should match its area under the density"

### Step 4 — Check the area under the density is 1 by a Riemann sum

The integral sign means "slice the interval into slivers, add up height × tiny width". We do it literally: 2,000 slivers, each of height 2 and width 0.00025. Their areas should sum to exactly 1 — the sum-to-1 rule, continuous edition.

In [ ]:
n_slices = 2_000
edges = np.linspace(0, 0.5, n_slices + 1)
slice_width = edges[1] - edges[0]
density_values = np.full(n_slices, 2.0)
total_area = np.sum(density_values * slice_width)

print(f"{n_slices} slivers of (height 2) x (width {slice_width:.5f}) add up to {total_area:.6f}")

assert abs(total_area - 1.0) < 1e-9, "the area under any density must be exactly 1"

## Several random things at once

The lesson's box of 100 raffle tickets, as a joint table (each cell is count/100):

|          | $y=1$ | $y=2$ | $y=3$ |
|----------|-------|-------|-------|
| $x=0$    | 0.10  | 0.20  | 0.10  |
| $x=1$    | 0.30  | 0.10  | 0.20  |

Three operations to verify: **marginals** are row/column sums ($p(x) = \sum_y p(x,y)$), **conditionals** are rescaled slices ($p(y \mid x) = p(x,y)/p(x)$), and **independence** would mean $p(x,y) = p(x)\,p(y)$ in every cell.

### Step 5 — Build the joint table and take marginals

The joint table becomes a 2 × 3 numpy array. `P.sum(axis=1)` adds across each row (summing away y) to give p(x); `P.sum(axis=0)` adds down each column (summing away x) to give p(y). Hand values to match: p(x) = (0.40, 0.60) and p(y) = (0.40, 0.30, 0.30).

In [ ]:
P = np.array([
    [0.10, 0.20, 0.10],   # x = 0 row
    [0.30, 0.10, 0.20],   # x = 1 row
])
xvals = np.array([0.0, 1.0])
yvals = np.array([1.0, 2.0, 3.0])

px = P.sum(axis=1)
py = P.sum(axis=0)

print("joint table P:")
print(P)
print("marginal p(x) (row sums)   :", px)
print("marginal p(y) (column sums):", py)
print(f"grand total of all cells   : {P.sum():.2f}")

assert abs(P.sum() - 1.0) < 1e-12, "a joint table must sum to 1 over all cells"
assert np.allclose(px, [0.40, 0.60]), "row sums should match the hand-computed marginal p(x)"
assert np.allclose(py, [0.40, 0.30, 0.30]), "column sums should match the hand-computed marginal p(y)"

### Step 6 — Slice and rescale: conditionals

Conditioning on x = 0 keeps only the first row and rescales it by its sum, 0.40. The lesson's hand answer: (0.25, 0.50, 0.25). For x = 1 the hand answer is (1/2, 1/6, 1/3). Both rescaled slices must sum to 1 — a conditional is a full-fledged PMF.

In [ ]:
cond_y_given_x0 = P[0] / px[0]
cond_y_given_x1 = P[1] / px[1]

print("p(y | x=0) =", np.round(cond_y_given_x0, 4), "  (hand: 0.25, 0.50, 0.25)")
print("p(y | x=1) =", np.round(cond_y_given_x1, 4), "  (hand: 1/2, 1/6, 1/3)")
print(f"sums: {cond_y_given_x0.sum():.4f} and {cond_y_given_x1.sum():.4f}")

assert abs(cond_y_given_x0.sum() - 1.0) < 1e-12, "a conditional must sum to 1"
assert abs(cond_y_given_x1.sum() - 1.0) < 1e-12, "a conditional must sum to 1"
assert np.allclose(cond_y_given_x0, [0.25, 0.50, 0.25]), "should match the hand computation"

### Step 7 — Test independence

If X and Y were independent, the joint table would equal the *outer product* of the marginals: every cell would be p(x) times p(y). `np.outer(px, py)` builds that hypothetical table. Compare it to the real one: cell (0, 0) would be 0.40 × 0.40 = 0.16, but the actual joint says 0.10. Knowing x genuinely changes the odds of y — these variables are dependent.

In [ ]:
independent_table = np.outer(px, py)
gap = np.abs(P - independent_table).max()

print("the table IF x and y were independent (outer product of marginals):")
print(np.round(independent_table, 3))
print("the actual table:")
print(P)
print(f"largest cell-by-cell difference: {gap:.3f}")

assert gap > 0.05, "the tables clearly differ: X and Y are dependent"

## Bayes' rule: flipping the direction of knowledge

$$p(x \mid y) = \frac{p(y \mid x)\; p(x)}{p(y)}$$

The lesson's medical test: a disease hits 1% of the population, the test catches 99% of the sick, and falsely alarms on 5% of the healthy. Counting 10,000 people gave 99 true positives and 495 false positives, so

$$p(\text{sick} \mid \text{positive}) = \frac{99}{594} = \frac{1}{6} \approx 0.167.$$

If Bayes' rule really is a counting argument in disguise, a *simulation* of 10,000 random people must land near the same number.

### Step 8 — Simulate 10,000 patients

Each person: sick with probability 0.01; then the test fires with probability 0.99 if sick, 0.05 if healthy. We then look only at people who tested positive and ask what fraction are sick — conditioning implemented as literal filtering. With 10,000 people, chance gives roughly ±0.02 of wiggle, so we allow a modest tolerance.

In [ ]:
n_people = 10_000
sick = rng.random(n_people) < 0.01
test_roll = rng.random(n_people)
positive = np.where(sick, test_roll < 0.99, test_roll < 0.05)

n_positive = int(positive.sum())
n_sick_and_positive = int((sick & positive).sum())
simulated = n_sick_and_positive / n_positive
hand_answer = 99 / 594

print(f"people: {n_people:,}   sick: {int(sick.sum())}   positive tests: {n_positive}")
print(f"simulated   p(sick | positive) = {simulated:.4f}")
print(f"hand answer p(sick | positive) = 99/594 = {hand_answer:.4f}")

assert abs(simulated - hand_answer) < 0.06, "the simulation should land near the counting answer"

### Step 9 — Tighten the check with a million patients

The Step 2 lesson applies here too: more samples, less wiggle. With 1,000,000 people the empirical answer should pin down 1/6 to about two decimal places.

In [ ]:
n_people_big = 1_000_000
sick_big = rng.random(n_people_big) < 0.01
test_roll_big = rng.random(n_people_big)
positive_big = np.where(sick_big, test_roll_big < 0.99, test_roll_big < 0.05)
simulated_big = (sick_big & positive_big).sum() / positive_big.sum()

print(f"with {n_people_big:,} people: p(sick | positive) = {simulated_big:.4f}   (hand: {hand_answer:.4f})")

assert abs(simulated_big - hand_answer) < 0.01, "a million patients should agree with Bayes to ~2 decimals"

## Expectation: the long-run average

$$\mathbb{E}[X] = \sum_x x\, p(x) \qquad \mathbb{E}[f(X)] = \sum_x f(x)\, p(x) \qquad \mathbb{E}[aX + bY] = a\,\mathbb{E}[X] + b\,\mathbb{E}[Y]$$

Three claims to verify on real numbers: the ticket table gives $\mathbb{E}[X] = 0.6$ and $\mathbb{E}[Y] = 1.9$; **linearity** holds *even though X and Y are dependent*; and **LOTUS** gives $\mathbb{E}[X^2] = 91/6 \approx 15.17$ for a die — which is *not* $(\mathbb{E}[X])^2 = 12.25$.

### Step 10 — Compute E[X], E[Y], and E[X+Y] exactly from the table

E[X] and E[Y] come from the marginals (value times probability, summed). E[X+Y] is computed the honest brute-force way: build the 2 × 3 grid of all possible sums x + y, weight each by its joint probability, and add all six terms. Linearity predicts the two routes agree — with no independence anywhere in sight.

In [ ]:
EX = float(xvals @ px)
EY = float(yvals @ py)
sum_grid = xvals[:, None] + yvals[None, :]
EXplusY = float(np.sum(sum_grid * P))

print(f"E[X] = {EX:.2f}   E[Y] = {EY:.2f}   E[X] + E[Y] = {EX + EY:.2f}")
print(f"E[X + Y] computed cell by cell from the joint table = {EXplusY:.2f}")

assert abs(EXplusY - (EX + EY)) < 1e-12, "linearity holds even though X and Y are dependent"

### Step 11 — Verify LOTUS by simulation, and dodge the classic trap

LOTUS: to average X², square each value *first*, then weight by probability: (1 + 4 + 9 + 16 + 25 + 36)/6 = 91/6. We check it against the empirical mean of 200,000 squared die rolls. The trap: squaring the *average* gives 3.5² = 12.25, a genuinely different number.

In [ ]:
n_rolls_big = 200_000
rolls_big = rng.integers(1, 7, size=n_rolls_big)
lotus_exact = np.sum(values ** 2 * (1 / 6))
empirical_mean_of_squares = (rolls_big.astype(float) ** 2).mean()
square_of_mean = rolls_big.mean() ** 2

print(f"LOTUS exact E[X^2] = 91/6 = {lotus_exact:.4f}")
print(f"empirical mean of squared rolls = {empirical_mean_of_squares:.4f}")
print(f"(E[X])^2, for contrast = {square_of_mean:.4f}  -- not the same thing")

assert abs(empirical_mean_of_squares - lotus_exact) < 0.1, "the sample average of X^2 should approach the LOTUS value"
assert lotus_exact - square_of_mean > 2.5, "E[X^2] and (E[X])^2 genuinely differ"

## Variance: the typical spread

$$\mathrm{Var}(X) = \mathbb{E}[X^2] - \mu^2 \qquad \mathrm{Var}(aX+b) = a^2\,\mathrm{Var}(X) \qquad \mathrm{Var}(X+Y) = \mathrm{Var}(X) + \mathrm{Var}(Y) \ \text{ (independent only!)}$$

The lesson's hand numbers for the ticket table: $\mathrm{Var}(X) = 0.24$, $\mathrm{Var}(Y) = 0.69$. Drawn **independently**, the variance of the sum should be $0.93$. Drawn **together from the joint table** (dependent), the lesson computed $\mathrm{Var}(X+Y) = 0.85 \ne 0.93$ — the addition rule must visibly fail. Both predictions get tested, plus a second dependent pair built a different way.

### Step 12 — Independent draws: variances add

Draw X from its marginal and Y from its marginal with two *separate* calls to the random generator — that makes them independent by construction, whatever the joint table said. Prediction: Var(X + Y) ≈ Var(X) + Var(Y) ≈ 0.93.

In [ ]:
n_draws = 400_000
x_ind = rng.choice(xvals, size=n_draws, p=px)
y_ind = rng.choice(yvals, size=n_draws, p=py)

var_sum_ind = (x_ind + y_ind).var()
sum_of_vars_ind = x_ind.var() + y_ind.var()

print(f"independent draws:  Var(X + Y)      = {var_sum_ind:.4f}")
print(f"                    Var(X) + Var(Y) = {sum_of_vars_ind:.4f}")
print("hand values: Var(X) = 0.24, Var(Y) = 0.69, sum = 0.93")

assert abs(var_sum_ind - sum_of_vars_ind) < 0.02, "for independent draws the variances add"
assert abs(var_sum_ind - 0.93) < 0.02, "and the total should match the hand value 0.93"

# The rule underneath it all: for independent draws, the average of the product
# equals the product of the averages.
product_average = (x_ind * y_ind).mean()
average_product = x_ind.mean() * y_ind.mean()

print(f"average of product : {product_average:.4f}")
print(f"product of averages: {average_product:.4f}")

assert abs(product_average - average_product) < 0.05


### Step 13 — Dependent draws from the joint table: variances do NOT add

Now draw *pairs* from the joint table itself: pick one of the 6 cells with the joint probabilities, then read off its x and y. This reproduces the dependence. Prediction from the lesson: E[X+Y] is still 2.50 (linearity survives dependence), but Var(X+Y) = 0.85, short of 0.93 by exactly the cross term 2 Cov = −0.08.

In [ ]:
cell_index = rng.choice(6, size=n_draws, p=P.flatten())
x_dep = xvals[cell_index // 3]
y_dep = yvals[cell_index % 3]

mean_sum_dep = (x_dep + y_dep).mean()
var_sum_dep = (x_dep + y_dep).var()
sum_of_vars_dep = x_dep.var() + y_dep.var()

print("dependent draws (pairs from the joint table):")
print(f"  E[X + Y]        = {mean_sum_dep:.4f}   (hand: 2.50 -- linearity survives dependence)")
print(f"  Var(X + Y)      = {var_sum_dep:.4f}   (hand: 0.85)")
print(f"  Var(X) + Var(Y) = {sum_of_vars_dep:.4f}   (hand: 0.93 -- NOT the same)")

assert abs(mean_sum_dep - 2.50) < 0.01, "linearity of expectation needs no independence"
assert abs(var_sum_dep - 0.85) < 0.02, "the true Var(X+Y) for the dependent pair is 0.85"
assert sum_of_vars_dep - var_sum_dep > 0.04, "for this dependent pair, the variances visibly do NOT add"

### Step 14 — A second dependent pair: Y = X + noise

Build dependence another way: let Y literally contain X. Then X + Y = 2X + noise, so the scaling rule predicts Var(X + Y) = 4·Var(X) + Var(noise) = 5, while naive addition claims Var(X) + Var(Y) = 1 + 2 = 3. Sharing the same X makes the sum spread *more* than the rule would predict — positive covariance at work.

In [ ]:
x_base = rng.standard_normal(n_draws)
noise = rng.standard_normal(n_draws)
y_linked = x_base + noise

var_sum_linked = (x_base + y_linked).var()
naive_sum = x_base.var() + y_linked.var()

print(f"Y = X + noise:  Var(X + Y)      = {var_sum_linked:.3f}   (prediction: 4 + 1 = 5)")
print(f"                Var(X) + Var(Y) = {naive_sum:.3f}   (prediction: 1 + 2 = 3)")

assert abs(var_sum_linked - 5.0) < 0.1, "Var(2X + noise) = 4 Var(X) + Var(noise) = 5"
assert abs(naive_sum - 3.0) < 0.1, "naive addition would claim 3"
assert var_sum_linked - naive_sum > 1.0, "the positive covariance makes the sum spread MORE than naive addition predicts"

## Estimating expectations by sampling

The Monte-Carlo estimate $\bar{X}_n = \frac{1}{n}\sum_{i=1}^n X_i$ is centered on the truth, and the two variance rules combined to give its error:

$$\mathrm{Var}(\bar{X}_n) = \frac{\mathrm{Var}(X)}{n}$$

So the typical error shrinks like $1/\sqrt{n}$. For die rolls, $\mathrm{Var}(X) = 35/12 \approx 2.92$: a 100-roll average should typically sit within about $\sqrt{2.92/100} \approx 0.17$ of 3.5. Two experiments: watch one running average settle, then measure the error at several $n$ and compare to the formula.

### Step 15 — Watch the running average settle at 3.5

`np.cumsum(rolls) / (1, 2, 3, ...)` gives the average of the first n rolls, for every n at once. On a log-scale x-axis you can see the early chaos and the late calm — the law of large numbers as a picture.

In [ ]:
n_rolls_lln = 100_000
rolls_lln = rng.integers(1, 7, size=n_rolls_lln)
running_average = np.cumsum(rolls_lln) / np.arange(1, n_rolls_lln + 1)

plt.figure(figsize=(8, 4))
plt.plot(running_average, color="#4ea1ff", linewidth=1)
plt.axhline(3.5, color="#ff7b72", linestyle="--", label="E[X] = 3.5")
plt.xscale("log")
plt.xlabel("number of rolls averaged (log scale)")
plt.ylabel("running average")
plt.title("The running average of die rolls settles onto 3.5")
plt.legend()
plt.show()

print(f"average of {n_rolls_lln:,} rolls = {running_average[-1]:.4f}")

assert abs(running_average[-1] - 3.5) < 0.02, "after 100,000 rolls the average should be within 0.02 of 3.5"

### Step 16 — Measure how the error shrinks with n

For each n in {10, 100, 1000}: compute 300 independent n-roll averages and take their standard deviation — the *measured* typical error of an n-sample estimate. Compare against the *predicted* error sqrt(Var/n). On a log-log plot, both should fall along the same straight line of slope −1/2.

In [ ]:
true_var = 35 / 12
mc_sizes = np.array([10, 100, 1000])
n_repeats = 300
measured_stds = []
for size in mc_sizes:
    batch_means = rng.integers(1, 7, size=(n_repeats, size)).mean(axis=1)
    measured_stds.append(batch_means.std())
measured_stds = np.array(measured_stds)
predicted_stds = np.sqrt(true_var / mc_sizes)

plt.figure(figsize=(7, 4))
plt.loglog(mc_sizes, measured_stds, "o-", color="#4ea1ff", label="measured std of the n-sample average")
plt.loglog(mc_sizes, predicted_stds, "--", color="#ff7b72", label="predicted sqrt(Var/n)")
plt.xlabel("n (samples per average)")
plt.ylabel("std of the average")
plt.title("Monte-Carlo error shrinks like 1/sqrt(n)")
plt.legend()
plt.show()

print("n            :", mc_sizes)
print("measured std :", np.round(measured_stds, 4))
print("predicted std:", np.round(predicted_stds, 4))

for measured, predicted in zip(measured_stds, predicted_stds):
    assert abs(measured - predicted) / predicted < 0.25, "measured error should track sqrt(Var/n) within 25%"

This is why minibatch training works: the loss you *want* is an expectation over all data — uncomputable — and the loss you *compute* is a Monte-Carlo average over a few hundred samples: noisy (that is the wiggle in every loss curve) but centered on the truth. You will watch this happen while training a diffusion model in part 10.

## Practice

The same problems as the lesson. Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Using the ticket table `P` from Step 5, compute the conditional distribution $p(x \mid y{=}3)$ and check that it sums to 1. (Hand targets: $1/3$ and $2/3$.)

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- The conditional lives in the $y=3$ column: joints $0.10$ and $0.20$.
- The marginal is the column sum: $p(y{=}3) = 0.10 + 0.20 = 0.30$.
- Rescale the slice: $0.10/0.30 = 1/3$ and $0.20/0.30 = 2/3$, which sum to 1.

```python
col = P[:, 2]                 # the y = 3 column
p_y3 = col.sum()
cond_x_given_y3 = col / p_y3

print("p(x | y=3) =", np.round(cond_x_given_y3, 4))
print("sum =", cond_x_given_y3.sum())
```

**Answer:** $p(x{=}0 \mid y{=}3) = 1/3$, $p(x{=}1 \mid y{=}3) = 2/3$.

</details>

**Problem 2.** Spam filter: 20% of email is spam; the word "free" appears in 60% of spam and 5% of legitimate mail. An email contains "free" — what is the probability it is spam? Count a population of 10,000 emails by hand, then confirm by simulation.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- 2,000 spam and 8,000 legitimate emails.
- Spam with "free": $0.60 \times 2{,}000 = 1{,}200$. Legitimate with "free": $0.05 \times 8{,}000 = 400$.
- Among the $1{,}600$ emails with "free", the spam fraction is $1{,}200/1{,}600 = 0.75$.
- Formula route: $\frac{0.60 \times 0.20}{0.60 \times 0.20 + 0.05 \times 0.80} = \frac{0.12}{0.16} = 0.75$.

```python
n_email = 1_000_000
is_spam = rng.random(n_email) < 0.20
word_roll = rng.random(n_email)
has_free = np.where(is_spam, word_roll < 0.60, word_roll < 0.05)
p_spam_given_free = (is_spam & has_free).sum() / has_free.sum()

print(f"simulated p(spam | 'free') = {p_spam_given_free:.4f}   (hand: 0.75)")
```

**Answer:** $0.75$ — much higher than the medical test's $1/6$, because the prior (20% spam) is not tiny.

</details>

**Problem 3.** $X$ takes the values $-1, 0, 2$ with probabilities $0.2, 0.5, 0.3$. Compute $\mathbb{E}[X]$ and $\mathrm{Var}(X)$ by hand, then confirm with numpy.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- $\mathbb{E}[X] = (-1)(0.2) + (0)(0.5) + (2)(0.3) = -0.2 + 0 + 0.6 = 0.4$.
- LOTUS: $\mathbb{E}[X^2] = (1)(0.2) + (0)(0.5) + (4)(0.3) = 1.4$.
- Shortcut: $\mathrm{Var}(X) = 1.4 - 0.4^2 = 1.4 - 0.16 = 1.24$.

```python
vals = np.array([-1.0, 0.0, 2.0])
probs = np.array([0.2, 0.5, 0.3])
mean_x = vals @ probs
var_x = (vals ** 2) @ probs - mean_x ** 2

print(f"E[X] = {mean_x:.2f}   Var(X) = {var_x:.2f}")
```

**Answer:** $\mathbb{E}[X] = 0.4$, $\mathrm{Var}(X) = 1.24$.

</details>

**Problem 4.** For the same $X$, compute $\mathrm{Var}(3X - 2)$ and $\mathbb{E}[3X - 2]$ using the rules — then confirm by transforming samples.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- $\mathrm{Var}(aX + b) = a^2\,\mathrm{Var}(X)$ with $a = 3$: $9 \times 1.24 = 11.16$. The shift $-2$ does not change spread.
- Linearity: $\mathbb{E}[3X - 2] = 3(0.4) - 2 = -0.8$. The mean *does* feel the shift.

```python
draws = rng.choice(vals, size=500_000, p=probs)
transformed = 3 * draws - 2

print(f"empirical E[3X-2]   = {transformed.mean():.3f}   (rule: -0.8)")
print(f"empirical Var(3X-2) = {transformed.var():.3f}   (rule: 11.16)")
```

**Answer:** $\mathrm{Var}(3X-2) = 11.16$, $\mathbb{E}[3X-2] = -0.8$.

</details>

**Problem 5.** A random variable is uniform on $[0, 1/10]$. Find its density, verify the total area is 1, compute $P(0.02 \le X \le 0.05)$, and explain why a density of 10 breaks no rule.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Flat height $c$ on a base of width $1/10$ needs area 1: $c \times \tfrac{1}{10} = 1$, so $c = 10$.
- $P(0.02 \le X \le 0.05) = 10 \times 0.03 = 0.3$.
- Density is a **rate** (probability per unit length). Only areas are probabilities, and no area under this curve can exceed $10 \times \tfrac{1}{10} = 1$.

```python
u_samples = rng.uniform(0, 0.1, size=200_000)
frac_in_window = ((u_samples >= 0.02) & (u_samples <= 0.05)).mean()

print(f"fraction in [0.02, 0.05] = {frac_in_window:.4f}   (area prediction: 0.3)")
```

**Answer:** density $= 10$ on $[0, 1/10]$; total area $= 1$; $P = 0.3$; the height 10 is a rate, not a probability.

</details>

**Problem 6.** Let $X$ be a fair coin (0 or 1) and $Y = X$ an exact copy. Compute $\mathrm{Var}(X)$, $\mathrm{Var}(Y)$, and $\mathrm{Var}(X+Y)$, and show the variances do not add. Which term in the lesson's derivation is responsible?

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- $\mathbb{E}[X] = 0.5$, $\mathbb{E}[X^2] = 0.5$, so $\mathrm{Var}(X) = 0.5 - 0.25 = 0.25 = \mathrm{Var}(Y)$.
- $X + Y = 2X$, and the scaling rule gives $\mathrm{Var}(2X) = 4 \times 0.25 = 1$.
- Naive addition claims $0.5$. The gap $0.5$ is the cross term $2\,\mathrm{Cov}(X,Y) = 2\,\mathrm{Var}(X) = 0.5$, which only independence would have killed.

```python
coin = (rng.random(500_000) < 0.5).astype(float)
copy = coin.copy()

print(f"Var(X)     = {coin.var():.4f}   (hand: 0.25)")
print(f"Var(X + Y) = {(coin + copy).var():.4f}   (hand: 1.0, NOT 0.5)")
```

**Answer:** $\mathrm{Var}(X+Y) = 1 \ne 0.5$; the covariance cross term supplies the missing $0.5$.

</details>

**Problem 7.** For one fair-die roll, use LOTUS to compute $\mathbb{E}[(X-1)^2]$, and show it differs from $(\mathbb{E}[X]-1)^2$.

In [ ]:
# Your turn:

<details><summary>Show worked solution</summary>

- Apply $f(x) = (x-1)^2$ to each face: $0, 1, 4, 9, 16, 25$.
- Sum $= 55$; divide by 6: $\mathbb{E}[(X-1)^2] = 55/6 \approx 9.17$.
- The other order: $(\mathbb{E}[X]-1)^2 = (2.5)^2 = 6.25$. Different — the function is applied *before* averaging in LOTUS.

```python
faces = np.arange(1, 7)
lotus_value = np.mean((faces - 1) ** 2)
other_order = (faces.mean() - 1) ** 2

print(f"E[(X-1)^2] = {lotus_value:.4f}   (55/6 = 9.1667)")
print(f"(E[X]-1)^2 = {other_order:.4f}   -- not the same")
```

**Answer:** $\mathbb{E}[(X-1)^2] = 55/6 \approx 9.17 \ne 6.25$.

</details>

## Wrap-up

Everything the lesson derived was verified numerically: empirical frequencies converge to the PMF; a density of 2 integrates to area 1 and predicts interval probabilities; marginals and conditionals fall out of the joint table exactly as the slice-and-rescale recipe says; a simulation of 10,000 patients reproduces the Bayes answer 1/6; linearity of expectation held even for a dependent pair, while variance addition held for independent draws and visibly failed for two different dependent constructions; and the Monte-Carlo error shrank like 1/sqrt(n), right on the predicted line.

Next, Part 3 puts these rules to work on the single most important distribution in the course: the Gaussian — the noise that diffusion models add, remove, and reason about.